# Лабораторная работа 5. ML-подходы для временных рядов

**Цель.** Сформировать лаговые признаки, сравнить Random Forest с базовым прогнозом и изучить поиск аномалий.

Ноутбук оформлен как самостоятельная версия учебной работы. Входные файлы, если они нужны для расчётов, лежат рядом с ноутбуком или скачиваются отдельной ячейкой.


## Ход работы

Ниже последовательно выполняются подготовка данных, построение графиков, расчёт признаков или моделей и краткая интерпретация результата. Случайные процедуры фиксируются через seed, чтобы результаты можно было повторить.


In [ ]:
import random
import numpy as np

random.seed(42)
np.random.seed(42)


In [ ]:
from matplotlib import pyplot as plt
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_percentage_error, mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor
from numpy import asarray

# Загружаем данные
passengers = pd.read_csv('passengers.csv')
passengers['Month'] = pd.to_datetime(passengers['Month'])
df = passengers.set_index('Month').sort_index()

# Создаем выборку с 12 лагами
n_lags = 12
cols = []
for i in range(1, n_lags + 1):
    cols.append(df.shift(i))

# Добавляем целевую переменную
cols.append(df.shift(0))

# Удаляем NaN
agg = pd.concat(cols, axis=1)
agg.dropna(inplace=True)
list_for_forecast = agg.values

print(f"Размер выборки: {len(list_for_forecast)}")
print(f"Количество признаков (лагов): {n_lags}")

# Прогноз с помощью Random Forest
def random_forest_forecast(train, testX):
    train = asarray(train)
    trainX, trainy = train[:, :-1], train[:, -1]
    model = RandomForestRegressor(n_estimators=1000, random_state=42)
    model.fit(trainX, trainy)
    yhat = model.predict([testX])
    return yhat[0]

# Разделяем на train и test
predictions = list()
size = int(len(list_for_forecast) * 0.66)
train, test = list_for_forecast[0:size], list_for_forecast[size:len(list_for_forecast)]
history = [x for x in train]

# Прогнозы
for i in range(len(test)):
    testX, testy = test[i, :-1], test[i, -1]
    yhat = random_forest_forecast(history, testX)
    predictions.append(yhat)
    history.append(test[i])

# Метрики
print("\nRandom Forest metrics (12 lags):")
print("RMSE:", np.sqrt(mean_squared_error(test[:, -1], predictions)))
print("MAPE:", mean_absolute_percentage_error(test[:, -1], predictions))
print("MAE:", mean_absolute_error(test[:, -1], predictions))
print("R2:", r2_score(test[:, -1], predictions))

# Визуализация
fig = plt.figure(figsize=(20, 9))
plt.plot(test[:, -1], label='Original data', linewidth=2)
plt.plot(predictions, label='Random Forest (12 lags)', linewidth=2)
plt.legend(fontsize="15")
plt.title('Airline passengers by month - 12 lags', fontsize="20")
plt.ylabel('Total passengers', fontsize="20")
plt.xlabel('Month', fontsize="20")
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
from sklearn.metrics import mean_absolute_error, r2_score
import numpy as np

# Naive Forecast
def naive_forecast(train, test):
    predictions = []
    last_value = train[-1, -1]

    for i in range(len(test)):
        if i == 0:
            yhat = last_value
        else:
            yhat = test[i-1, -1]
        predictions.append(yhat)

    return predictions

# Разделяем данные
n_lags = 12
cols = []
for i in range(1, n_lags + 1):
    cols.append(df.shift(i))
cols.append(df.shift(0))

agg = pd.concat(cols, axis=1)
agg.dropna(inplace=True)
list_for_forecast = agg.values

size = int(len(list_for_forecast) * 0.66)
train, test = list_for_forecast[0:size], list_for_forecast[size:len(list_for_forecast)]

# Прогнозы наивной модели
naive_predictions = naive_forecast(train, test)

# Метрики
print("Naive Forecast metrics:")
print("RMSE:", np.sqrt(mean_squared_error(test[:, -1], naive_predictions)))
print("MAPE:", mean_absolute_percentage_error(test[:, -1], naive_predictions))
print("MAE:", mean_absolute_error(test[:, -1], naive_predictions))
print("R2:", r2_score(test[:, -1], naive_predictions))

# Сравнение метрик
print("\n" + "="*60)
print("СРАВНЕНИЕ МОДЕЛЕЙ")
print("="*60)
print(f"{'Метрика':<10} | {'Naive':<15} | {'RF (12 lags)':<15} | {'Улучшение':<15}")
print("-"*60)
print(f"{'MAE':<10} | {mean_absolute_error(test[:, -1], naive_predictions):<15.2f} | {mean_absolute_error(test[:, -1], predictions):<15.2f} | {'RF лучше' if mean_absolute_error(test[:, -1], predictions) < mean_absolute_error(test[:, -1], naive_predictions) else 'Naive лучше':<15}")
print(f"{'R2':<10} | {r2_score(test[:, -1], naive_predictions):<15.4f} | {r2_score(test[:, -1], predictions):<15.4f} | {'RF лучше' if r2_score(test[:, -1], predictions) > r2_score(test[:, -1], naive_predictions) else 'Naive лучше':<15}")
print("="*60)

# Визуализация
fig = plt.figure(figsize=(20, 9))
plt.plot(test[:, -1], label='Original data', linewidth=2, color='black')
plt.plot(naive_predictions, label='Naive Forecast', linewidth=2, linestyle='--')
plt.plot(predictions, label='Random Forest (12 lags)', linewidth=2)
plt.legend(fontsize="15")
plt.title('Comparison: Naive vs Random Forest', fontsize="20")
plt.ylabel('Total passengers', fontsize="20")
plt.xlabel('Month', fontsize="20")
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
from sklearn.ensemble import IsolationForest
import plotly.express as px

# Загружаем данные
pas = pd.read_csv('passengers.csv')
pas['Month'] = pd.to_datetime(pas['Month'])

# Модель с contamination=0.05
model = IsolationForest(contamination=0.05, random_state=42)
model.fit(pas[['Passengers']])

# Метка аномалий
pas['outliers'] = pd.Series(model.predict(pas[['Passengers']])).apply(lambda x: 'yes' if (x == -1) else 'no')

# Найденные аномалии
print("Найденные аномалии (contamination=0.05):")
print(pas.query('outliers=="yes"'))
print(f"\nВсего аномалий: {len(pas.query('outliers=="yes"'))} из {len(pas)} ({len(pas.query('outliers=="yes"'))/len(pas)*100:.1f}%)")

# Визуализация
fig = px.scatter(pas, x='Month', y='Passengers', color='outliers',
                 color_discrete_map={'no': '#636efa', 'yes': '#EF553B'},
                 title='Anomaly Detection with IsolationForest (contamination=0.05)')
fig.update_xaxes(
    rangeslider_visible=True,
)
fig.update_layout(
    font=dict(size=14),
    title_font_size=20,
    hovermode='x unified'
)
fig.show()

# Сравнение с оригинальным параметром
print("\n" + "="*60)
print("СРАВНЕНИЕ ПАРАМЕТРОВ CONTAMINATION")
print("="*60)

# Оригинальная модель (0.004)
model_orig = IsolationForest(contamination=0.004, random_state=42)
model_orig.fit(pas[['Passengers']])
pas['outliers_orig'] = pd.Series(model_orig.predict(pas[['Passengers']])).apply(lambda x: 'yes' if (x == -1) else 'no')

print(f"contamination=0.004: {len(pas.query('outliers_orig=="yes"'))} аномалий")
print(f"contamination=0.05:  {len(pas.query('outliers=="yes"'))} аномалий")
print("="*60)

# Анализ аномалий
print("\nДетальный анализ аномалий при contamination=0.05:")
anomalies = pas.query('outliers=="yes"')
for idx, row in anomalies.iterrows():
    print(f"Дата: {row['Month']}, Пассажиры: {row['Passengers']}")

## Вывод

В работе показан полный путь от подготовки временного ряда до визуализации, расчёта признаков или построения модели. Основные результаты следует оценивать по графикам и метрикам, полученным при запуске ноутбука.
